# Phase 4: Label Correspondence (1-NN based)

The current CV data are built so that source and target do not overlap (InChIKey overlap 0), so labels of the same molecule cannot be compared directly.
Instead, this summarises the results of `01_model/11_nn_label_correspondence.ipynb`: the label correlation between each target molecule and its most similar source molecule (ECFP4 Tanimoto 1-NN).

- `cross` = target ↔ nearest source (in vitro ↔ in vivo correspondence)
- `within_target` = target ↔ nearest other target (upper bound expected from structural similarity within the same endpoint)
- `relative_to_within` = cross ρ / within ρ (closer to 1 means the endpoints correspond)

**Run notebook 11 first.** The representative threshold is Tanimoto ≥ 0.4, where all three endpoints have enough pairs (n ≥ 85).

In [1]:
import os
import json
import numpy as np
import pandas as pd

ROOT = r'..'
RESULTS_DIR = os.path.join(ROOT, 'benchmark', 'results')
NN_CSV = os.path.join(ROOT, 'results', 'nn_label_correspondence', 'nn_label_correlation.csv')
PROPERTIES = ['clearance', 'half_life', 'fu']
MAIN_T = 0.4

res = pd.read_csv(NN_CSV)

In [2]:
def f(x):
    return None if pd.isna(x) else float(x)

label_correspondence_results = {}
for prop in PROPERTIES:
    r = res[res['property'] == prop]
    cross = r[(r['kind'] == 'cross') & (r['source_set'] == 'all')].set_index('threshold')
    within = r[r['kind'] == 'within_target'].set_index('threshold')
    by_t = {}
    for t in cross.index:
        c, w = cross.loc[t], within.loc[t]
        rel = c['spearman_rho'] / w['spearman_rho'] if pd.notna(c['spearman_rho']) and pd.notna(w['spearman_rho']) else np.nan
        by_t[str(t)] = {
            'n_pairs': int(c['n']),
            'spearman_rho': f(c['spearman_rho']), 'rho_ci': [f(c['rho_lo']), f(c['rho_hi'])],
            'pearson_r': f(c['pearson_r']),
            'null_rho_95': f(c['null_rho_95']),
            'above_null': bool(pd.notna(c['spearman_rho']) and c['spearman_rho'] > c['null_rho_95']),
            'within_target_rho': f(w['spearman_rho']), 'within_target_n': int(w['n']),
            'relative_to_within': f(rel),
        }
    main = by_t[str(MAIN_T)]
    label_correspondence_results[prop] = {
        'method': '1-NN label correlation (ECFP4 Tanimoto)',
        'main_threshold': MAIN_T,
        **{k: main[k] for k in ['n_pairs', 'spearman_rho', 'rho_ci', 'null_rho_95', 'above_null',
                                'within_target_rho', 'relative_to_within']},
        'by_threshold': by_t,
    }

pd.DataFrame({p: {k: v for k, v in d.items() if k != 'by_threshold'} for p, d in label_correspondence_results.items()}).T

,method,main_threshold,n_pairs,spearman_rho,rho_ci,null_rho_95,above_null,within_target_rho,relative_to_within
clearance,1-NN label correlation (ECFP4 Tanimoto),0.4,139,0.193205,"[0.0236474090423901, 0.3498866732655849]",0.169618,True,0.468442,0.412441
half_life,1-NN label correlation (ECFP4 Tanimoto),0.4,125,0.046824,"[-0.1336016541926815, 0.2302392827600287]",0.181553,False,0.5467,0.085649
fu,1-NN label correlation (ECFP4 Tanimoto),0.4,85,0.698974,"[0.5533565127233594, 0.8285560566040474]",0.213405,True,0.636897,1.097467


In [3]:
with open(os.path.join(RESULTS_DIR, 'phase_04_label_correspondence.json'), 'w') as fh:
    json.dump(label_correspondence_results, fh, indent=2)